# Train FPL Points Prediction Model
This notebook focuses on training a model to predict FPL points using the features from the previous notebooks.

## 1. Include required libraries

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import joblib

## 2. Read Data
We read the CSV files generated by earlier steps into Spark dataframes:
* features/XXX.csv

In [2]:
# Load the dataset
input_filepath = "../data/processed/features/20250406_162326/features.parquet"
data = pd.read_parquet(input_filepath)

# Show data
data.head()

,fixture_code,team_code,opponent_team_code,kickoff_time,season_code,player_name,player_code,team_name,opponent_team_name,position,...,recent_team_clean_sheets,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false
0,2292813,14,54,2022-08-06 11:30:00,2022_23,James Milner,15157,Liverpool,Fulham,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,0,1
1,2292825,14,31,2022-08-15 19:00:00,2022_23,James Milner,15157,Liverpool,Crystal Palace,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,1,0
2,2292836,14,1,2022-08-22 19:00:00,2022_23,James Milner,15157,Liverpool,Man Utd,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,0,1
3,2292845,14,91,2022-08-27 14:00:00,2022_23,James Milner,15157,Liverpool,Bournemouth,MID,...,NaN,NaN,NaN,NaN,0,0,1,0,1,0
4,2292857,14,4,2022-08-31 19:00:00,2022_23,James Milner,15157,Liverpool,Newcastle,MID,...,1.0,NaN,214.0,NaN,0,0,1,0,1,0


## 3. Select Features
We select features for training the model.

In [3]:
selected_features = [
    "recent_goals_scored",
    "long_term_goals_scored",
    "recent_goals_conceded",
    "long_term_goals_conceded",
    "recent_assists",
    "long_term_assists",
    "recent_expected_assists",
    "long_term_expected_assists",
    "recent_expected_goal_involvements",
    "long_term_expected_goal_involvements",
    "recent_expected_goals",
    "long_term_expected_goals",
    "recent_expected_goals_conceded",
    "long_term_expected_goals_conceded",
    "recent_penalties_missed",
    "long_term_penalties_missed",
    "recent_penalties_saved",
    "long_term_penalties_saved",
    "recent_saves",
    "long_term_saves",
    "recent_bps",
    "long_term_bps",
    "recent_minutes",
    "long_term_minutes",
    "recent_yellow_cards",
    "long_term_yellow_cards",
    "recent_red_cards",
    "long_term_red_cards",
    "recent_own_goals",
    "long_term_own_goals",
    "recent_starts",
    "long_term_starts",
    "recent_influence",
    "long_term_influence",
    "recent_creativity",
    "long_term_creativity",
    "recent_threat",
    "long_term_threat",
    "recent_ict_index",
    "long_term_ict_index",
    "recent_clean_sheets",
    "long_term_clean_sheets",
    "recent_total_points",
    "long_term_total_points",
    "recent_team_goals_scored",
    "long_term_team_goals_scored",
    "recent_team_goals_conceded",
    "long_term_team_goals_conceded",
    "recent_team_assists",
    "long_term_team_assists",
    "recent_team_expected_assists",
    "long_term_team_expected_assists",
    "recent_team_expected_goal_involvements",
    "long_term_team_expected_goal_involvements",
    "recent_team_expected_goals",
    "long_term_team_expected_goals",
    "recent_team_expected_goals_conceded",
    "long_term_team_expected_goals_conceded",
    "recent_team_penalties_missed",
    "long_term_team_penalties_missed",
    "recent_team_penalties_saved",
    "long_term_team_penalties_saved",
    "recent_team_saves",
    "long_term_team_saves",
    "recent_team_bps",
    "long_term_team_bps",
    "recent_team_yellow_cards",
    "long_term_team_yellow_cards",
    "recent_team_red_cards",
    "long_term_team_red_cards",
    "recent_team_own_goals",
    "long_term_team_own_goals",
    "recent_team_clean_sheets",
    "long_term_team_clean_sheets",
    "recent_team_total_points",
    "long_term_team_total_points",
    "position_GK",
    "position_DEF",
    "position_MID",
    "position_FWD",
    "was_home_true",
    "was_home_false",
    "total_points",
]
    
# Filter the columns in the DataFrame
training_data = data[selected_features]

# Show the training data
training_data.head()

,recent_goals_scored,long_term_goals_scored,recent_goals_conceded,long_term_goals_conceded,recent_assists,long_term_assists,recent_expected_assists,long_term_expected_assists,recent_expected_goal_involvements,long_term_expected_goal_involvements,...,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false,total_points
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,0,1,0,0,1,1.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,0,1,0,1,0,5.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,0,1,0,0,1,2.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,0,1,0,1,0,1.0
4,0.0,NaN,4.0,NaN,1.0,NaN,0.0,NaN,0.0,NaN,...,NaN,214.0,NaN,0,0,1,0,1,0,1.0


## 4. Data Cleaning
We'll handle data cleaning tasks such as dealing with missing values.

In [4]:
# Check missing values
print("Missing values before handling:\n", training_data.isnull().sum())

# Drop rows with missing values
training_data = training_data.dropna()

# Verify no missing values
print("Missing values after handling:\n", training_data.isnull().sum())

# Show the data
training_data.head()

Missing values before handling:
 recent_goals_scored         10142
long_term_goals_scored      17545
recent_goals_conceded       10142
long_term_goals_conceded    17545
recent_assists              10142
                            ...  
position_MID                    0
position_FWD                    0
was_home_true                   0
was_home_false                  0
total_points                 5608
Length: 83, dtype: int64
Missing values after handling:
 recent_goals_scored         0
long_term_goals_scored      0
recent_goals_conceded       0
long_term_goals_conceded    0
recent_assists              0
                           ..
position_MID                0
position_FWD                0
was_home_true               0
was_home_false              0
total_points                0
Length: 83, dtype: int64


,recent_goals_scored,long_term_goals_scored,recent_goals_conceded,long_term_goals_conceded,recent_assists,long_term_assists,recent_expected_assists,long_term_expected_assists,recent_expected_goal_involvements,long_term_expected_goal_involvements,...,long_term_team_clean_sheets,recent_team_total_points,long_term_team_total_points,position_GK,position_DEF,position_MID,position_FWD,was_home_true,was_home_false,total_points
12,0.0,0.0,2.0,5.0,0.0,1.0,0.00000,0.0,0.00000,0.0,...,2.0,185.0,371.0,0,0,1,0,0,1,0.0
13,0.0,0.0,2.0,4.0,0.0,1.0,0.00000,0.0,0.00000,0.0,...,3.0,168.0,397.0,0,0,1,0,1,0,1.0
14,0.0,0.0,2.0,3.0,0.0,0.0,0.01529,0.0,0.01529,0.0,...,4.0,156.0,428.0,0,0,1,0,0,1,0.0
15,0.0,0.0,1.0,2.0,0.0,0.0,0.01529,0.0,0.01529,0.0,...,4.0,186.0,427.0,0,0,1,0,1,0,0.0
16,0.0,0.0,0.0,3.0,0.0,0.0,0.01529,0.0,0.01529,0.0,...,3.0,189.0,342.0,0,0,1,0,0,1,0.0


## 4. Train Model
We'll train the model.

In [5]:
# Prepare the data for the model
X = training_data.drop('total_points', axis=1)
y = training_data['total_points']

# Define cross-validation parameters
cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize models
linear_model = LinearRegression()

# Perform cross-validation for Linear Regression
linear_mse_scores = -cross_val_score(linear_model, X, y, cv=cv, scoring='neg_mean_squared_error')
linear_r2_scores = cross_val_score(linear_model, X, y, cv=cv, scoring='r2')

# Print the cross-validation results
print("Linear Regression Cross-Validation Results:")
print(f"Average MSE: {np.mean(linear_mse_scores)}")
print(f"Average R-squared: {np.mean(linear_r2_scores)}")

Linear Regression Cross-Validation Results:
Average MSE: 3.7583289862111213
Average R-squared: 0.32635229661287013


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Prepare the data for the model
X = training_data.drop('total_points', axis=1)
y = training_data['total_points']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Regressor model
rf_model = RandomForestRegressor(random_state=42, n_estimators=10)

# Train the model on the training data
rf_model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_rf = rf_model.predict(X_val)

# Calculate the Mean Squared Error (MSE)
rf_mse = mean_squared_error(y_val, y_pred_rf)

# Calculate the R-squared (R^2) score
rf_r2 = r2_score(y_val, y_pred_rf)

# Print the validation results
print("\nRandom Forest Regression Validation Results:")
print(f"Mean Squared Error: {rf_mse}")
print(f"R-squared: {rf_r2}")


Random Forest Regression Validation Results:
Mean Squared Error: 3.622079249945411
R-squared: 0.33216644046270016


In [7]:
# Retrain the best model on the entire dataset
# For this example, let's assume Random Forest performed better
final_model = rf_model
final_model.fit(X, y)

RandomForestRegressor(n_estimators=10, random_state=42)

## 6. Export Model
Finally, we'll export the trained model.

In [8]:
from datetime import datetime
import os

# Define output directory
# We define the directory where the processed data will be saved.
data_source = "random_forest"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/model/{data_source}/{current_datetime}"
filename = 'model.joblib'
filepath = os.path.join(output_dir, filename)

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save the model
joblib.dump(final_model, filepath)

print(f"Model training complete. Model saved to {output_dir}")

Model training complete. Model saved to ../data/model/random_forest/20250406_162635
